# Gemini 라벨링

- 샘플링된 면접 데이터(2,000건)에 4개 항목(두괄식/논리구조/핵심 키워드/분량) 라벨 생성
- 중간에 끊겨도 이어서 실행 가능 (체크포인트 방식)
- 실행 순서: 위에서부터 셀 순서대로 실행

## 1. 패키지 설치

In [ ]:
!pip install -q google-generativeai

## 2. Google Drive 마운트
입력/출력 파일을 Drive에 저장하고 싶으면 실행. 로컬 세션 파일로만 쓸 거면 건너뛰어도 됨.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 3. API 키 입력

In [ ]:
from google.colab import userdata
# GEMINI_API_KEY = userdata.get('GOOGLE_API_KEY')
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY_1')

## 4. 임포트 & 설정값

In [ ]:
import json
import re
import time
from pathlib import Path

import google.generativeai as genai

INPUT_PATH = "/content/drive/MyDrive/AIHub_면접데이터/sampled_dataset.jsonl"     # 샘플링된 입력 데이터
OUTPUT_PATH = "/content/drive/MyDrive/AIHub_면접데이터/labeled_dataset.jsonl"    # 라벨링 결과 저장 경로
LIMIT = None   # 테스트할 때는 예: 20  (앞 20건만 처리). 전체 돌릴 땐 None

MODEL_NAME = "gemini-3.1-flash-lite"
REQUEST_DELAY_SEC = 4.5          # 무료 티어 RPM 제한 고려 (초당 요청 과다 방지). 429 뜨면 늘릴 것
MAX_RETRIES = 3
RETRY_BACKOFF_SEC = 15

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


## 5. 라벨링 프롬프트

In [ ]:
LABELING_SYSTEM_PROMPT = """당신은 취업 면접 답변을 채점하는 평가자입니다. 주어진 질문과 답변을 보고
아래 4가지 항목을 채점하여 반드시 JSON 형식으로만 출력하세요. 다른 설명은 절대 추가하지 마세요.

[평가 항목]

1. 두괄식 (headline)
- 답변의 첫 문장(또는 첫 1~2문장)에 결론/핵심 주장이 명확히 드러나는가
- true / false 로 판정
- 판단 기준: 첫 문장이 "저는 ~했습니다", "제 강점은 ~입니다" 처럼 결론형으로 시작하면 true
- 배경 설명이나 상황 묘사로 먼저 시작하면 false

2. 논리 구조 (logic_structure)
- 결론 → 근거 → (마무리) 흐름이 자연스럽게 이어지는가
- 1~5점 척도, 아래 기준으로 판단하고 애매해도 중간값(2~3)으로만 몰지 말 것. 기준에 맞으면 4점, 5점도 적극적으로 부여할 것
  - 1점: 근거가 거의 없거나 질문과 무관한 내용으로 채워짐
  - 2점: 근거는 있으나 매우 약하거나 반복적이고, 결론과의 연결이 느슨함
  - 3점: 근거가 있고 결론과 어느 정도 연결되나, 흐름이 산만하거나 마무리가 없음
  - 4점: 결론-근거 연결이 명확함. 마무리가 다소 약하거나 부연설명이 조금 늘어짐
  - 5점: 결론-근거-마무리가 모두 명확하고 군더더기 없이 이어짐
- 구어체 추임새("어", "음", "이제" 등)는 이 항목 채점에서 무시할 것 — 이런 표현이 많다고 논리 구조 점수를 깎지 말고, 오직 결론-근거-마무리의 연결 자체만 볼 것
- 근거가 결론과 무관하거나 비약이 있으면 감점

3. 키워드 포함 (keywords)
- 주어진 직무(job)와 관련된 핵심 역량/기술 키워드가 답변에 포함되어 있는가
- 주의: job은 "지원 회사의 업종 카테고리"이며 지원자의 실제 지원 직무와 다를 수 있음 (예: job="ICT"이지만 실제로는 인사총무 지원). 반드시 질문(question)과 답변(answer) 내용을 먼저 읽고, 그 안에서 지원자가 언급하는 실제 직무/역할을 우선 기준으로 삼을 것. job은 참고용 힌트일 뿐, 답변 내용과 충돌하면 답변 내용을 따를 것
- 답변에서 실제 발견된 키워드 리스트를 추출 (found_keywords)
- 직무 관련이지만 답변에 없는 주요 키워드도 추정해서 리스트업 (missing_keywords, 최대 3개)

4. 분량 (length)
- 답변의 글자수(공백 포함)를 세어 적정(150~400자) 여부 판정
- char_count: 숫자
- is_appropriate: true/false

[출력 형식 - 반드시 이 JSON 스키마만 사용, 코드블록 없이 순수 JSON만]
{
  "headline": true,
  "logic_structure": 3,
  "logic_reason": "한 줄 이유",
  "found_keywords": ["키워드1"],
  "missing_keywords": ["키워드1"],
  "char_count": 0,
  "is_appropriate": true
}

[예시 1]
직무: 백엔드 개발자
질문: 팀 내 갈등이 생겼을 때 어떻게 해결했나요?
답변: 저는 협업 과정에서 의견 충돌이 생기면 먼저 상대방의 입장을 듣고 공통의 목표를 다시 확인하는 방식으로 해결합니다. 이전 프로젝트에서 API 설계 방식을 두고 팀원과 의견이 갈렸을 때, 각자의 장단점을 정리해 문서화하고 팀 회의에서 데이터 기반으로 결정했습니다. 그 결과 프로젝트 일정 지연 없이 마무리할 수 있었습니다.
출력:
{"headline": true, "logic_structure": 5, "logic_reason": "결론(해결 방식)-사례(API 설계 갈등)-결과(일정 준수) 흐름이 명확함", "found_keywords": ["협업", "API 설계"], "missing_keywords": ["데이터베이스", "성과"], "char_count": 178, "is_appropriate": true}

[예시 2]
직무: 백엔드 개발자
질문: 대용량 트래픽을 처리해본 경험이 있나요?
답변: 예전에 인턴십을 했었는데 그때 트래픽이 많은 서비스를 다뤄본 적이 있습니다. 처음엔 잘 몰라서 좀 헤맸는데 나중에 어찌저찌 해결은 했습니다.
출력:
{"headline": false, "logic_structure": 2, "logic_reason": "배경 설명만 있고 구체적 해결 방법이나 결과가 없음", "found_keywords": [], "missing_keywords": ["캐싱", "로드밸런싱", "DB 최적화"], "char_count": 89, "is_appropriate": false}

[예시 3]
직무: 마케팅
질문: 팀 프로젝트에서 갈등을 겪은 경험이 있나요?
답변: 네 있습니다. 협업 툴 도입 방식을 두고 팀원과 의견이 갈렸던 적이 있는데요, 각자 장단점을 정리해서 공유하고 팀 회의에서 결정했습니다. 다만 그 과정에서 의견 조율에 시간이 좀 걸렸던 점은 아쉬웠습니다.
출력:
{"headline": true, "logic_structure": 4, "logic_reason": "결론(갈등 경험 있음)-근거(협업 툴 의견 차이와 해결 과정)가 명확하게 연결되나, 마무리에서 아쉬운 점을 짧게 덧붙이는 데 그쳐 정리가 다소 약함", "found_keywords": ["협업", "의견 조율"], "missing_keywords": ["데이터 기반 의사결정", "성과"], "char_count": 112, "is_appropriate": false}

이제 아래 질문과 답변을 채점하세요.
"""

## 6. 헬퍼 함수

In [ ]:
def build_prompt(job: str, question: str, answer: str) -> str:
    # 직무/질문/답변을 받아 Gemini에 보낼 전체 프롬프트를 완성
    return (
        f"{LABELING_SYSTEM_PROMPT}\n\n"
        f"직무: {job}\n"
        f"질문: {question}\n"
        f"답변: {answer}\n"
        f"출력:"
    )


def extract_json(text: str) -> dict:
    # Gemini 응답에서 JSON만 뽑아냄 (```json 코드블록으로 감싸는 경우 대비)
    text = text.strip()
    text = re.sub(r"^```(json)?", "", text).strip()
    text = re.sub(r"```$", "", text).strip()
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        raise ValueError(f"JSON을 찾을 수 없음: {text[:200]}")
    return json.loads(match.group(0))


def load_done_ids(output_path: Path) -> set:
    # 이미 라벨링 완료된 id 목록 (재실행 시 이어서 하기 위함)
    done = set()
    if output_path.exists():
        with open(output_path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    row = json.loads(line)
                    done.add(row["id"])
                except (json.JSONDecodeError, KeyError):
                    continue
    return done


def label_one(model, job: str, question: str, answer: str) -> dict:
    # 답변 1건을 Gemini에 보내 라벨을 받아옴 (실패 시 재시도)
    prompt = build_prompt(job, question, answer)
    last_err = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = model.generate_content(
                prompt,
                generation_config={"temperature": 0.1},
            )
            return extract_json(response.text)
        except Exception as e:
            last_err = e
            wait = RETRY_BACKOFF_SEC * attempt
            print(f"  [재시도 {attempt}/{MAX_RETRIES}] 실패: {e} -> {wait}초 대기")
            time.sleep(wait)
    raise RuntimeError(f"최종 실패: {last_err}")

## 7. 모델 초기화
`INPUT_PATH`에 샘플링된 jsonl 파일이 있어야 함. 각 줄: `{"id": 1, "job": "...", "question": "...", "answer": "..."}`

In [ ]:
genai.configure(api_key=GEMINI_API_KEY)
model = genai.GenerativeModel(MODEL_NAME)
print("모델 준비 완료:", MODEL_NAME)

모델 준비 완료: gemini-3.1-flash-lite


## 8. 배치 실행
먼저 `LIMIT = 20` 정도로 테스트해보고, 이상 없으면 위 설정 셀에서 `LIMIT = None`으로 바꿔서 전체 실행 권장.

In [ ]:
input_path = Path(INPUT_PATH)
output_path = Path(OUTPUT_PATH)

with open(input_path, "r", encoding="utf-8") as f:
    rows = [json.loads(line) for line in f if line.strip()]
if LIMIT:
    rows = rows[:LIMIT]

done_ids = load_done_ids(output_path)
print(f"전체 {len(rows)}건 중 이미 완료된 {len(done_ids)}건은 건너뜀")

fail_count = 0
with open(output_path, "a", encoding="utf-8") as out_f:
    for i, row in enumerate(rows, 1):
        row_id = row.get("id", i)
        if row_id in done_ids:
            continue

        job = row.get("job", "")
        question = row.get("question", "")
        answer = row.get("answer", "")

        print(f"[{i}/{len(rows)}] id={row_id} 처리 중...")
        try:
            labels = label_one(model, job, question, answer)
            result = {**row, "labels": labels}
            out_f.write(json.dumps(result, ensure_ascii=False) + "\n")
            out_f.flush()
        except Exception as e:
            fail_count += 1
            print(f"  실패 (건너뜀): {e}")
            with open(output_path.with_suffix(".failed.jsonl"), "a", encoding="utf-8") as fail_f:
                fail_f.write(json.dumps({**row, "error": str(e)}, ensure_ascii=False) + "\n")

        time.sleep(REQUEST_DELAY_SEC)

print(f"\n완료. 실패 {fail_count}건은 {output_path.with_suffix('.failed.jsonl')} 참고")

전체 1999건 중 이미 완료된 1634건은 건너뜀
[1635/1999] id=1635 처리 중...
[1636/1999] id=1636 처리 중...
[1637/1999] id=1637 처리 중...
[1638/1999] id=1638 처리 중...
[1639/1999] id=1639 처리 중...
[1640/1999] id=1640 처리 중...
[1641/1999] id=1641 처리 중...
[1642/1999] id=1642 처리 중...
[1643/1999] id=1643 처리 중...
[1644/1999] id=1644 처리 중...
[1645/1999] id=1645 처리 중...
[1646/1999] id=1646 처리 중...
[1647/1999] id=1647 처리 중...
[1648/1999] id=1648 처리 중...
[1649/1999] id=1649 처리 중...
[1650/1999] id=1650 처리 중...
[1651/1999] id=1651 처리 중...
[1652/1999] id=1652 처리 중...
[1653/1999] id=1653 처리 중...
[1654/1999] id=1654 처리 중...
[1655/1999] id=1655 처리 중...
[1656/1999] id=1656 처리 중...
[1657/1999] id=1657 처리 중...
[1658/1999] id=1658 처리 중...
[1659/1999] id=1659 처리 중...
[1660/1999] id=1660 처리 중...
[1661/1999] id=1661 처리 중...
[1662/1999] id=1662 처리 중...
[1663/1999] id=1663 처리 중...
[1664/1999] id=1664 처리 중...
[1665/1999] id=1665 처리 중...
[1666/1999] id=1666 처리 중...
[1667/1999] id=1667 처리 중...
[1668/1999] id=1668 처리 중...
[1669/1999] id=1669

ERROR:tornado.access:503 POST /v1beta/models/gemini-3.1-flash-lite:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 2944.52ms


[1723/1999] id=1723 처리 중...
[1724/1999] id=1724 처리 중...
[1725/1999] id=1725 처리 중...
[1726/1999] id=1726 처리 중...
[1727/1999] id=1727 처리 중...
[1728/1999] id=1728 처리 중...
[1729/1999] id=1729 처리 중...
[1730/1999] id=1730 처리 중...
[1731/1999] id=1731 처리 중...
[1732/1999] id=1732 처리 중...
[1733/1999] id=1733 처리 중...
[1734/1999] id=1734 처리 중...
[1735/1999] id=1735 처리 중...
[1736/1999] id=1736 처리 중...
[1737/1999] id=1737 처리 중...
[1738/1999] id=1738 처리 중...
[1739/1999] id=1739 처리 중...
[1740/1999] id=1740 처리 중...
[1741/1999] id=1741 처리 중...
[1742/1999] id=1742 처리 중...
[1743/1999] id=1743 처리 중...
[1744/1999] id=1744 처리 중...
[1745/1999] id=1745 처리 중...
[1746/1999] id=1746 처리 중...
[1747/1999] id=1747 처리 중...
[1748/1999] id=1748 처리 중...
[1749/1999] id=1749 처리 중...
[1750/1999] id=1750 처리 중...
[1751/1999] id=1751 처리 중...
[1752/1999] id=1752 처리 중...
[1753/1999] id=1753 처리 중...
[1754/1999] id=1754 처리 중...
[1755/1999] id=1755 처리 중...
[1756/1999] id=1756 처리 중...
[1757/1999] id=1757 처리 중...
[1758/1999] id=1758 

## 9. (선택) 결과 미리 확인

In [ ]:
import pandas as pd

with open(output_path, "r", encoding="utf-8") as f:
    labeled = [json.loads(line) for line in f]

df = pd.json_normalize(labeled)
print(f"라벨링 완료 건수: {len(df)}")
df.head()

라벨링 완료 건수: 1999


,id,job,question,answer,meta.occupation_code,meta.gender,meta.experience,meta.file_path,labels.headline,labels.logic_structure,labels.logic_reason,labels.found_keywords,labels.missing_keywords,labels.char_count,labels.is_appropriate
0,1,경영/사무,귀하께서는 전 직장에 약 한 삼 년 동안 근무를 하셨는데 전 직장에서 기억에 남는 ...,어 예전에 중국 여행 상품을 개발했던 것이 생각납니다. 어 당시에 중국 여행 상품을...,01.Management,Male,Experienced,/content/drive/MyDrive/AIHub_면접데이터/Training/02...,False,3,"프로젝트의 배경과 문제 상황, 해결 과정이 구체적으로 서술되어 있으나, 답변의 첫 ...","[상품 개발, 프로젝트, 컨텐츠]","[문제 해결 능력, 기획력, 고객 만족]",458,False
1,2,경영/사무,직전 회사를 그만두신 이유가 있으실 겁니다 혹시 동일한 이유로 현 직장도 그만둘 가...,이전 직장을 그만둔 특별한 이유가 있다면 전 직장에 대표 이사나 경영진들이 직원들을...,01.Management,Male,Experienced,/content/drive/MyDrive/AIHub_면접데이터/Training/02...,False,2,"질문에 대한 직접적인 답변보다 전 직장에 대한 부정적인 감정 표출이 앞서며, 현 직...","[경영진, 직원 관리]","[조직 문화, 가치관, 커뮤니케이션]",338,True
2,3,경영/사무,지금까지 인생을 살면서 후회되는 일이 있다면 어떤 일인지 저에게 설명 부탁드리도록 ...,저는 살면서 후회를 잘 하지 않는데요. 가장 후회되는 일이 하나 있다면 제가 저의 ...,01.Management,Female,New,/content/drive/MyDrive/AIHub_면접데이터/Training/02...,True,5,"후회되는 사건을 명확히 밝히고, 그 이유와 배경, 그리고 깨달음과 변화된 가치관까지...","[양육, 애착 관계, 가치관]","[우선순위, 일과 삶의 균형, 책임감]",368,True
3,4,경영/사무,지원자님께서 이 직무를 수행하는 데 있어서 본인만의 장점이 있다면 그 이유와 함께 ...,저는 직무를 수행함에 있어 꼼꼼함이라는 장점이 있습니다. 어떤 일을 하든 실수를 하...,01.Management,Male,Experienced,/content/drive/MyDrive/AIHub_면접데이터/Training/02...,True,3,"꼼꼼함이라는 장점을 제시하고 구체적인 확인 리스트 활용법을 근거로 들었으나, 문장이...","[꼼꼼함, 데이터, 오타]","[업무 효율성, 정확성, 데이터 관리]",268,True
4,5,경영/사무,지원자님께서는 평소 스트레스를 어떻게 해소하십니까 자신만의 스트레스 해소 방법이 있...,저는 스트레스 해소를 하기 위해 산책을 하고 있습니다. 스트레스가 쌓인다고 생각이 ...,01.Management,Female,New,/content/drive/MyDrive/AIHub_면접데이터/Training/02...,True,4,"결론(산책)을 먼저 제시하고, 그 이유(공기 변화, 생각 정리)와 효과를 논리적으로...","[스트레스 해소, 생각 정리, 자기 성찰]","[업무 효율, 멘탈 관리, 회복 탄력성]",228,True


## 10. (선택) 라벨 분포 확인
두괄식/논리구조 점수가 한쪽으로 쏠려있진 않은지 체크

In [ ]:
print(df["labels.headline"].value_counts())
print()
print(df["labels.logic_structure"].value_counts().sort_index())
print()
print(df["labels.is_appropriate"].value_counts())

labels.headline
False    1078
True      921
Name: count, dtype: int64

labels.logic_structure
1     89
2    870
3    858
4    176
5      6
Name: count, dtype: int64

labels.is_appropriate
True     1460
False     539
Name: count, dtype: int64
